In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from PDF")
print("\n--- First page preview (first 500 chars)")
print(pages[0].page_content[:500])

C:\Users\ngass\AppData\Local\Temp\ipykernel_42284\1745050327.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\ngass\OneDrive\Documents\Python\AGENTIC-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from PDF

--- First page preview (first 500 chars)
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, # ~150 words per chunk
    chunk_overlap=100, # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "], # tries paragraph -> line -> sentence -> word
)

chunks = splitter.split_documents(pages)

len(chunks)

37

In [8]:
chunks[0]

Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1')

In [ ]:
len(chunks[0].page_content)

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

C:\Users\ngass\OneDrive\Documents\Python\AGENTIC-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ngass\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3247.44it/s]


Vector store ready. 37 vectors stored.


In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

test_query = "What is VolTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam



In [12]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# helper: join retrieved chunks into a single context string
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


SYSTEM_PROMPT = """\
You are a helpful telecom assistant
Answer the question using only the context provided below.
If the context does not content enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# --- LLM via GROQ API
llm = ChatGroq(
    model = "qwen/qwen3-32b",
    temperature=0,
    reasoning_format="parsed"
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.


In [13]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print(f"A: {chain.invoke(question)}")

Q: How does international roaming work and what charges should I expect?

A: International roaming allows your device to connect to partner networks in foreign countries when you're outside your home network's coverage. Here's how it works and what charges to expect:

### **How It Works**  
1. **Authentication**: When you travel abroad, your device connects to a partner network in the visited country. The visited network authenticates your subscription via signaling protocols (SS7/Diameter) and requests authorization from your home network.  
2. **Traffic Tunneling**: All voice, data, and SMS traffic is routed back to your home network for billing, which may introduce latency compared to local network usage.  

### **Charges**  
- **Roaming Zones**:  
  - **Zone A** (EU, UK, Australia, New Zealand): Lowest roaming rates.  
  - **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
  - **Zone C** (Rest of the world): Highest per-MB and per-minute charges.  
- **Bundles**: Always 